# Security Scanning

Run `sneppx-analyze` on C/C++/CUDA source, then exercise the S0
post-quantum crypto bindings (Dilithium, Kyber, SecureAllocator).

In [ ]:
import json, subprocess
from SneppX_ALG import Ed25519, Dilithium, KyberKEM, sha256,
import SneppX_ALG as S
HAS_C = S._HAS_C_BACKEND
print('C backend:', HAS_C)

## 1. Run sneppx-analyze (static scan, safe to run anywhere)

In [ ]:
# Scans algorithms/hss/core for buffer overflows, NULL derefs, etc.
proc = subprocess.run(
    ['sneppx-analyze', 'scan', 'algorithms/hss/core/',
     '--format', 'c', '--json'],
    capture_output=True, text=True,
)
report = json.loads(proc.stdout) if proc.stdout else {'findings': []}
print('findings:', len(report.get('findings', [])))

## 2. Dilithium post-quantum signing (S0)

In [ ]:
msg = b'SNEPPX-Algo release v1.2.0'
if HAS_C:
    dil = Dilithium(level=3)            # FIPS 204 ML-DSA-87
    sig = dil.sign(msg)
    assert dil.verify(msg, sig)
    print('Dilithium-3 OK, sig len:', len(sig))
else:
    print('C backend required - build: cmake --build build --target neural_security_c')

## 3. Kyber KEM (S0)

In [ ]:
if HAS_C:
    kem = KyberKEM(k=768)
    ct, sk = kem.encapsulate()
    ss = kem.decapsulate(ct)
    assert ss == sk
    print('Kyber-768 KEM OK')

## 4. Hash + secure memory attestation

In [ ]:
print('sha256:', sha256(b'model-weights.bin').hex()[:32], '...')
if HAS_C:
    from SneppX_ALG import SecureAllocator, StackCanary, MemoryLeakDetector
    buf = SecureAllocator(8192).alloc()    # guard pages + mlock
    StackCanary().install()
    print('leaks:', MemoryLeakDetector().scan())